# SEC/CCM Unified Runner

This notebook bootstraps the canonical script implementation in `src/thesis_pkg/notebooks_and_scripts/sec_ccm_unified_runner.py`.

For Google Colab it mounts Drive, reuses `NLP_THESIS_REPO_ROOT` when already set, and otherwise clones the repo into `/content/NLP_Thesis` before running the canonical script. Optional runner overrides can be supplied with `SEC_CCM_*` environment variables in the setup cell.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_ENV_VAR = "NLP_THESIS_REPO_ROOT"
GIT_URL_ENV_VAR = "NLP_THESIS_GIT_URL"
DEFAULT_GIT_URL = "https://github.com/Erew42/NLP_Thesis.git"
CLONE_ROOT = Path("/content/NLP_Thesis")


def _find_repo_root() -> Path | None:
    candidates: list[Path] = []
    env_root = os.environ.get(REPO_ENV_VAR)
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    if IN_COLAB:
        candidates.extend(
            [
                CLONE_ROOT,
                Path("/content/drive/MyDrive/NLP_Thesis"),
                Path("/content/drive/My Drive/NLP_Thesis"),
            ]
        )

    seen: set[Path] = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "src" / "thesis_pkg" / "pipeline.py").exists():
            return candidate
    return None


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

repo_root = _find_repo_root()
if repo_root is None and IN_COLAB:
    git_url = os.environ.get(GIT_URL_ENV_VAR, DEFAULT_GIT_URL)
    if CLONE_ROOT.exists() and not (CLONE_ROOT / "src" / "thesis_pkg" / "pipeline.py").exists():
        raise FileExistsError(
            f"{CLONE_ROOT} exists but does not look like the NLP_Thesis repo. Remove it or set {REPO_ENV_VAR}."
        )
    if not CLONE_ROOT.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", git_url, str(CLONE_ROOT)])
    repo_root = CLONE_ROOT

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the NLP_Thesis checkout. Run this notebook from the repo root, set NLP_THESIS_REPO_ROOT, or use Colab so the notebook can clone the repo automatically."
    )

# Optional overrides before the final cell runs:
# os.environ["SEC_CCM_DATA_PROFILE"] = "DRIVE_FULL"
# os.environ["SEC_CCM_WORK_ROOT"] = "/content/drive/MyDrive/Data_LM"
# os.environ["SEC_CCM_RUN_ROOT"] = "/content/drive/MyDrive/Data_LM/results/sec_ccm_unified_runner"
# os.environ["SEC_CCM_FF48_SICCODES_PATH"] = "/content/NLP_Thesis/full_data_run/LM2011_additional_data/FF_Siccodes_48_Industries.txt"

os.environ[REPO_ENV_VAR] = str(repo_root)
src = repo_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB and os.environ.get("SEC_CCM_SKIP_PIP_INSTALL", "0") != "1":
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--editable", str(repo_root)]
    )

print(
    {
        "IN_COLAB": IN_COLAB,
        "repo_root": str(repo_root),
        "src_exists": src.exists(),
        "data_profile": os.environ.get("SEC_CCM_DATA_PROFILE", "<default>"),
        "work_root": os.environ.get("SEC_CCM_WORK_ROOT", "<default>"),
        "run_root": os.environ.get("SEC_CCM_RUN_ROOT", "<default>"),
    }
)


In [ ]:
from thesis_pkg.notebooks_and_scripts.sec_ccm_unified_runner import main

main()
